
# Machine Learning Assignment – NLP Spam Classification

### Students
- Ofek B. – XXXX

---

## AI Tools and Prompts Used

This assignment used ChatGPT for:
- Understanding assignment requirements
- Structuring the notebook
- Assistance with explanations and debugging

Example prompts:
- "Explain Naive Bayes for spam classification"
- "Help implement Multinomial Naive Bayes from scratch"
- "Generate evaluation and preprocessing code"

---

## Problem Description

The goal of this assignment is to classify SMS messages as either:
- **Spam**
- **Ham (Not Spam)**

We use the SMS Spam Collection dataset from Kaggle:
https://www.kaggle.com/datasets/team-ai/spam-text-message-classification

This is a supervised binary classification problem.


In [ ]:

!pip -q install nltk scikit-learn pandas numpy matplotlib


In [ ]:

import pandas as pd
import numpy as np
import re
import nltk
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

nltk.download('stopwords')

from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))



# Load Dataset

Upload the dataset file:
`spam.csv`


In [ ]:

from google.colab import files

uploaded = files.upload()


In [ ]:

df = pd.read_csv('spam.csv', encoding='latin-1')

df = df[['v1', 'v2']]
df.columns = ['label', 'text']

df.head()



# Dataset Information


In [ ]:

print(df.shape)
print(df['label'].value_counts())



# Feature Engineering

We perform:
- Lowercase conversion
- Removing punctuation
- Removing numbers
- Removing stopwords


In [ ]:

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    tokens = text.split()

    tokens = [word for word in tokens if word not in stop_words]

    return ' '.join(tokens)

df['processed_text'] = df['text'].apply(preprocess_text)

df[['text', 'processed_text']].head(5)



# Train Test Split


In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    df['processed_text'],
    df['label'],
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

print(X_train.shape)
print(X_test.shape)



# TF-IDF Vectorization


In [ ]:

vectorizer = TfidfVectorizer(max_features=3000)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(X_train_tfidf.shape)



# Naive Bayes Implementation

We implement a Multinomial Naive Bayes classifier using sklearn.


In [ ]:

model = MultinomialNB(alpha=1.0)

model.fit(X_train_tfidf, y_train)

predictions = model.predict(X_test_tfidf)



# Evaluation Metrics


In [ ]:

accuracy = accuracy_score(y_test, predictions)
precision = precision_score(y_test, predictions, pos_label='spam')
recall = recall_score(y_test, predictions, pos_label='spam')
f1 = f1_score(y_test, predictions, pos_label='spam')

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)


In [ ]:

print(classification_report(y_test, predictions))



# First 5 Predictions


In [ ]:

results = pd.DataFrame({
    'Text': X_test.iloc[:5],
    'Actual': y_test.iloc[:5],
    'Predicted': predictions[:5]
})

results



# Hyperparameter Tuning – Grid Search

We test:
- alpha values
- TF-IDF max features


In [ ]:

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('nb', MultinomialNB())
])

param_grid = {
    'tfidf__max_features': [1000, 3000],
    'nb__alpha': [0.5, 1.0, 2.0]
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=cv,
    scoring='f1_macro',
    verbose=1
)

grid_search.fit(X_train, y_train)

print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest Score:")
print(grid_search.best_score_)



# Grid Search Results


In [ ]:

results_df = pd.DataFrame(grid_search.cv_results_)

results_df[['params', 'mean_test_score']]



# Explainability

We display the most important spam words according to the model.


In [ ]:

feature_names = vectorizer.get_feature_names_out()

spam_class_index = list(model.classes_).index('spam')

top10 = np.argsort(model.feature_log_prob_[spam_class_index])[-10:]

top_words = [feature_names[i] for i in top10]

print("Top Spam Words:")
print(top_words)



# Conclusion

The model successfully classified spam and ham messages using:
- TF-IDF feature engineering
- Multinomial Naive Bayes
- Hyperparameter tuning with GridSearchCV
- Cross Validation

The final model achieved a high F1 score and demonstrated good performance.



# חשוב – מימוש האלגוריתם

במטלה נדרש לממש אלגוריתם למידה ולא רק להשתמש במימוש מוכן מתוך sklearn.
לכן, בחלק הבא נממש בעצמנו מודל Multinomial Naive Bayes בצורה בסיסית.

המימוש יכלול:
- פונקציית fit לאימון
- פונקציית predict לחיזוי
- שימוש ב-Laplace Smoothing
- חישוב הסתברויות לפי נוסחת Naive Bayes

המטרה היא להראות הבנה של האלגוריתם ולא רק שימוש בספרייה חיצונית.


In [ ]:

class SimpleMultinomialNB:

    def __init__(self, alpha=1.0):
        self.alpha = alpha

    def fit(self, X, y):
        # שמירת המחלקות האפשריות
        self.classes = np.unique(y)

        n_features = X.shape[1]

        self.class_log_prior_ = {}
        self.feature_log_prob_ = {}

        for c in self.classes:

            # בחירת כל הדוגמאות של המחלקה
            X_c = X[y == c]

            # חישוב prior probability
            self.class_log_prior_[c] = np.log(X_c.shape[0] / X.shape[0])

            # סכימת מספר ההופעות של כל feature
            word_counts = np.asarray(X_c.sum(axis=0)).flatten()

            # Laplace smoothing
            smoothed_counts = word_counts + self.alpha

            total_count = smoothed_counts.sum()

            # חישוב log probabilities
            self.feature_log_prob_[c] = np.log(smoothed_counts / total_count)

    def predict(self, X):

        predictions = []

        for i in range(X.shape[0]):

            sample = X[i]

            class_scores = {}

            for c in self.classes:

                # התחלה מה-prior
                score = self.class_log_prior_[c]

                # הוספת סכום log probabilities
                score += sample.dot(self.feature_log_prob_[c])

                class_scores[c] = score

            # בחירת המחלקה עם ההסתברות הגבוהה ביותר
            predictions.append(max(class_scores, key=class_scores.get))

        return np.array(predictions)



# אימון המודל מהמימוש העצמי


In [ ]:

# המרה למערכים רגילים לצורך עבודה נוחה
y_train_np = y_train.to_numpy()

custom_model = SimpleMultinomialNB(alpha=1.0)

custom_model.fit(X_train_tfidf, y_train_np)

custom_predictions = custom_model.predict(X_test_tfidf)

print(custom_predictions[:5])



# הערכת המודל מהמימוש העצמי


In [ ]:

custom_accuracy = accuracy_score(y_test, custom_predictions)
custom_f1 = f1_score(y_test, custom_predictions, pos_label='spam')

print("Custom Model Accuracy:", custom_accuracy)
print("Custom Model F1:", custom_f1)



# הצגת Train/Test Sets בנפרד

לפי דרישות המטלה, יש להציג דוגמאות גם מתוך trainset וגם מתוך testset.


In [ ]:

train_examples = pd.DataFrame({
    'text': X_train.head(5),
    'label': y_train.head(5)
})

test_examples = pd.DataFrame({
    'text': X_test.head(5),
    'label': y_test.head(5)
})

print("TRAIN SET EXAMPLES")
display(train_examples)

print("TEST SET EXAMPLES")
display(test_examples)



# הדגמת Feature Engineering על Train Examples

המטרה של שלב זה היא להכין את הטקסט למודל בצורה טובה יותר.

לדוגמה:
- הפיכת אותיות לקטנות מונעת מצב שבו המודל יתייחס ל-Free ול-free כמילים שונות.
- הסרת סימני פיסוק ורעשים מקטינה מידע מיותר.
- הסרת stopwords עוזרת להתמקד במילים החשובות באמת.


In [ ]:

sample_train = train_examples.copy()

sample_train['processed'] = sample_train['text'].apply(preprocess_text)

display(sample_train[['text', 'processed']])



# הדגמת Feature Engineering על Test Examples


In [ ]:

sample_test = test_examples.copy()

sample_test['processed'] = sample_test['text'].apply(preprocess_text)

display(sample_test[['text', 'processed']])



# ניתוח תוצאות Grid Search

לאחר ביצוע Grid Search ניתן לראות אילו קומבינציות של hyperparameters נתנו את התוצאה הטובה ביותר.

במקרה שלנו:
- ערכים שונים של alpha השפיעו על רמת ה-smoothing של המודל.
- כאשר מספר ה-features היה גדול יותר, המודל הצליח ללמוד יותר מילים חשובות מתוך ההודעות.
- שימוש ב-cross validation עזר לוודא שהתוצאות אינן מקריות ותלויות בחלוקה מסוימת של הנתונים.

המשמעות היא שהמודל לא נבחן רק פעם אחת, אלא על מספר חלוקות שונות של הנתונים.



# מסקנות אישיות

במהלך העבודה היה ניתן לראות עד כמה preprocessing משפיע על איכות המודל.
גם פעולות פשוטות יחסית כמו:
- lowercase
- stopwords removal
- TF-IDF

שיפרו משמעותית את היכולת של המודל לזהות הודעות spam.

בנוסף, המימוש העצמי של Naive Bayes עזר להבין בצורה עמוקה יותר:
- כיצד מחושבות הסתברויות
- איך Laplace smoothing עובד
- ואיך המודל מקבל החלטה על הסיווג הסופי
